# Inductive bias: MLP (static) vs MLP (window) vs Conv1D (window)

Binary time-series segmentation on 3-channel signals: the class changes over time inside a single sample. Each timestep has its own label (class A or class B).

Class B segments are marked in one of two ways:
- **Pattern A** (prob 0.8): mean of `signal_0`, `signal_1` shifts within the segment
- **Pattern B** (prob 0.9): `signal_2` rapidly oscillates within the segment

We will compare three per-timestep classifiers:
- **MLP (static)** sees one timestep at a time (input shape `(3,)`). No temporal context (blind to pattern B).
- **MLP (window)** sees a small fixed time window around a timestep, flattened to `(3*W,)`.
- **Conv1D** applies convs to the same `(3, W)` window.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
import pandas as pd

np.random.seed(0)
torch.manual_seed(0)

## Data

In [ ]:
_d = np.load("data/conv_data.npz")
X_train, Y_train = _d["X_train"], _d["Y_train"]
X_test,  Y_test  = _d["X_test"],  _d["Y_test"]
T = int(_d["T"]); N_SIGNALS = int(_d["N_SIGNALS"])

diag = {}
for k in [key[len("diag_"):-len("_X")] for key in _d.files
          if key.startswith("diag_") and key.endswith("_X")]:
    diag[k] = (_d[f"diag_{k}_X"], _d[f"diag_{k}_Y"])

print("train:", X_train.shape, Y_train.shape, "\ntest: ", X_test.shape, Y_test.shape)
for k, (Xd, Yd) in diag.items():
    print(f"  diag[{k}]: X={Xd.shape}  classB-fraction={Yd.mean():.3f}")

### Example signals

One sample from each diagnostic group with the per-timestep ground-truth class shaded in the background (blue = class A, red = class B).

In [ ]:
def shade_labels(ax, y, alpha=0.15):
    # paint contiguous class-B regions
    in_b = False
    start = 0
    for t in range(len(y) + 1):
        v = y[t] if t < len(y) else 0
        if v == 1 and not in_b:
            in_b = True; start = t
        elif v == 0 and in_b:
            ax.axvspan(start, t, color="red", alpha=alpha, linewidth=0)
            in_b = False

titles = {"only_A": "class B segments using Pattern A only (mean shift in signal_0/1)",
          "only_B": "class B segments using Pattern B only (oscillation in signal_2)",
          "both":   "class B segment using both patterns"}
fig, axes = plt.subplots(3, 1, figsize=(10, 5), sharex=True, constrained_layout=True)
rng = np.random.default_rng(123)
for ax, (k, (Xd, Yd)) in zip(axes, diag.items()):
    i = rng.integers(len(Xd))
    x, y = Xd[i], Yd[i]
    shade_labels(ax, y)
    for s in range(N_SIGNALS):
        ax.plot(x[s], label=f"signal_{s}", alpha=0.85, linewidth=2 if s == 2 else 0.75)
    ax.set_title(titles[k])
    ax.set_ylabel("signal$_{0,1,2}$")
    ax.legend(loc="upper right", fontsize=8, ncol=3)
axes[-1].set_xlabel("time")
fig.set_dpi(200)
plt.show()

## Models

**MLP (static)** maps one timestep `(3,)` to the class probability of that timestep.

**MLP (window)** flattens a `(3, W)` window to `(3*W,)` and applies a few fully-connected layers. Expressive enough to capture both patterns, but no translation symmetry.

**Conv1D** applies a couple of convs (no padding) to the `(3, W)` window which collapse it into a small temporal axis, then a linear head produces the final logit. Translation along the time axis is built in.

To get per-timestep predictions on a sample we pad it by `W//2` on each side and apply the window models to every centered window.

In [ ]:
WINDOW = 31

class MLPStatic(nn.Module):
    "single-timestep: input (3,), output class of that timestep"
    def __init__(self, hidden_dim=32, n_hidden_layers=2):
        super().__init__()
        layers = [nn.Linear(N_SIGNALS, hidden_dim), nn.ReLU()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):   # x: (B, 3)
        return self.net(x)

class MLP(nn.Module):
    "window-based: input (3,W), output class of center timestep"
    def __init__(self, window=WINDOW, hidden_dim=128, n_hidden_layers=2):
        super().__init__()
        in_dim = N_SIGNALS * window
        layers = [nn.Flatten(), nn.Linear(in_dim, hidden_dim), nn.ReLU()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):   # x: (B, 3, W)
        return self.net(x)

class Conv1DNet(nn.Module):
    "window-based: input (3,W), output class of center timestep"
    def __init__(self, window=WINDOW, channels=24, kernel_size=5, n_layers=2, hidden=32):
        super().__init__()
        layers = []
        in_c, t = N_SIGNALS, window
        for _ in range(n_layers):
            layers += [nn.Conv1d(in_c, channels, kernel_size=kernel_size), nn.ReLU()]
            in_c = channels
            t = t - kernel_size + 1
        assert t >= 1, "kernel_size * n_layers exceeds window"
        self.conv = nn.Sequential(*layers)
        self.flat_dim = channels * t
        self.head = nn.Sequential(nn.Linear(self.flat_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, 1))

    def forward(self, x):  # x: (B, 3, W)
        h = self.conv(x).flatten(1)
        return self.head(h)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ---------- training -----------------------------------------------------
def _windows_with_center_labels(X, Y, window=WINDOW):
    # X: (N,3,T), Y: (N,T) -> (M,3,W), (M,) labels at window center.
    N, _, T_ = X.shape
    half = window // 2
    Xp = np.pad(X, ((0, 0), (0, 0), (half, half)), mode="edge")
    wins = np.lib.stride_tricks.sliding_window_view(Xp, window_shape=window, axis=-1)  # (N,3,T_,W)
    wins = np.transpose(wins, (0, 2, 1, 3))  # (N, T_, 3, W)
    wins = wins.reshape(-1, N_SIGNALS, window)
    labels = Y.reshape(-1)
    return wins, labels

def train_window_model(model, X, Y, epochs=200, lr=1e-3, weight_decay=1e-4,
                       window=WINDOW, subsample=0.2, batch=512):
    "Train a model that maps (B,3,W) windows to center-timestep logits."
    torch.manual_seed(0)
    wins, lab = _windows_with_center_labels(X, Y, window=window)
    n = wins.shape[0]
    take = int(n * subsample)
    rng = np.random.default_rng(0)
    sel = rng.choice(n, size=take, replace=False)
    Xt = torch.tensor(wins[sel], dtype=torch.float32)
    yt = torch.tensor(lab[sel, None], dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    idx = np.arange(Xt.shape[0])
    for _ in range(epochs):
        np.random.shuffle(idx)
        for s in range(0, len(idx), batch):
            b = idx[s:s + batch]
            opt.zero_grad()
            loss_fn(model(Xt[b]), yt[b]).backward()
            opt.step()
    return model

def train_mlp_static(X, Y, epochs=200, lr=1e-2, weight_decay=1e-4,
                     hidden_dim=32, n_hidden_layers=2):
    torch.manual_seed(0)
    model = MLPStatic(hidden_dim, n_hidden_layers)
    Xf = torch.tensor(X.transpose(0, 2, 1).reshape(-1, N_SIGNALS), dtype=torch.float32)
    yf = torch.tensor(Y.reshape(-1, 1), dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    batch = 4096
    idx = np.arange(Xf.shape[0])
    for _ in range(epochs):
        np.random.shuffle(idx)
        for s in range(0, len(idx), batch):
            b = idx[s:s + batch]
            opt.zero_grad()
            loss_fn(model(Xf[b]), yf[b]).backward()
            opt.step()
    return model

def train_mlp(X, Y, hidden_dim=128, n_hidden_layers=2, window=WINDOW, **kw):
    torch.manual_seed(0)
    model = MLP(window=window, hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
    return train_window_model(model, X, Y, window=window, **kw)

def train_conv(X, Y, channels=8, kernel_size=5, n_layers=2, hidden=8, window=WINDOW, **kw):
    torch.manual_seed(0)
    model = Conv1DNet(window=window, channels=channels, kernel_size=kernel_size,
                      n_layers=n_layers, hidden=hidden)
    return train_window_model(model, X, Y, window=window, **kw)

# ---------- inference ----------------------------------------------------
@torch.no_grad()
def predict_mlp_static(model, X):
    # X: (N,3,T) -> per-timestep probs (N,T).
    N, _, T_ = X.shape
    Xf = torch.tensor(X.transpose(0, 2, 1).reshape(-1, N_SIGNALS), dtype=torch.float32)
    p = torch.sigmoid(model(Xf)).numpy().ravel().reshape(N, T_)
    return p

@torch.no_grad()
def _predict_window_model(model, X, window=WINDOW, batch=4096):
    # X: (N,3,T) -> per-timestep probs (N,T) by sliding the window.
    N, _, T_ = X.shape
    wins, _ = _windows_with_center_labels(X, np.zeros((N, T_), dtype=int), window=window)
    Xt = torch.tensor(wins, dtype=torch.float32)
    out = []
    for s in range(0, Xt.shape[0], batch):
        out.append(torch.sigmoid(model(Xt[s:s + batch])).numpy().ravel())
    return np.concatenate(out).reshape(N, T_)

predict_mlp = _predict_window_model
predict_conv = _predict_window_model

## Train and compare

In [ ]:
mlp_static = train_mlp_static(X_train, Y_train)
mlp = train_mlp(X_train, Y_train)
conv = train_conv(X_train, Y_train)

print(f"# trainable parameters:")
print(f"  MLP (static) : {count_params(mlp_static):,}")
print(f"  MLP (window) : {count_params(mlp):,}")
print(f"  Conv1D       : {count_params(conv):,}")

def per_step_acc(P, Y):
    return accuracy_score(Y.ravel(), (P.ravel() > 0.5).astype(int))

mlp_static_test = per_step_acc(predict_mlp_static(mlp_static, X_test), Y_test)
mlp_test = per_step_acc(predict_mlp(mlp, X_test), Y_test)
conv_test = per_step_acc(predict_conv(conv, X_test), Y_test)
print(f"\nPer-timestep accuracy:")
print(f"  MLP (static) : {mlp_static_test:.3f}")
print(f"  MLP (window) : {mlp_test:.3f}")
print(f"  Conv1D       : {conv_test:.3f}")

### (A) Prediction traces

Top row = true labels (no shading = class A, red shading = class B). Lower rows = predicted P(class B) over time for each model.

In [ ]:
def plot_trace(X_sample, Y_sample, p_static, p_mlp, p_conv, title=""):
    fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True, constrained_layout=True)
    # signals
    shade_labels(axes[0], Y_sample)
    for s in range(N_SIGNALS):
        axes[0].plot(X_sample[s], label=f"signal_{s}", alpha=0.85, linewidth=2 if s == 2 else 0.75)
    axes[0].legend(loc="upper right", fontsize=8, ncol=3)
    axes[0].set_title(title or "signals (red shading = true class B)")
    axes[0].set_ylabel("value")
    # MLP static
    shade_labels(axes[1], Y_sample)
    axes[1].plot(p_static, color="#9467bd", linewidth=1.2)
    axes[1].axhline(0.5, color="black", linestyle=":", linewidth=0.6)
    axes[1].set_ylim(-0.05, 1.05); axes[1].set_ylabel("MLP static P(B)")
    # MLP window
    shade_labels(axes[2], Y_sample)
    axes[2].plot(p_mlp, color="#d62728", linewidth=1.2)
    axes[2].axhline(0.5, color="black", linestyle=":", linewidth=0.6)
    axes[2].set_ylim(-0.05, 1.05); axes[2].set_ylabel("MLP window P(B)")
    # Conv
    shade_labels(axes[3], Y_sample)
    axes[3].plot(p_conv, color="#1f77b4", linewidth=1.2)
    axes[3].axhline(0.5, color="black", linestyle=":", linewidth=0.6)
    axes[3].set_ylim(-0.05, 1.05); axes[3].set_ylabel("Conv1D P(B)")
    axes[3].set_xlabel("time")
    plt.show()

# one trace per diagnostic group
for k, (Xd, Yd) in diag.items():
    i = 1
    p_s = predict_mlp_static(mlp_static, Xd[i:i + 1])[0]
    p_m = predict_mlp(mlp, Xd[i:i + 1])[0]
    p_c = predict_conv(conv, Xd[i:i + 1])[0]
    plot_trace(Xd[i], Yd[i], p_s, p_m, p_c, title=f"diagnostic group: {k}")

## (B) Per-pattern accuracy

Let's check how often the model correctly classifies a class-B timestep for each type of pattern.

In [ ]:
groups = list(diag.keys())
static_b, mlp_b, conv_b = [], [], []
for g in groups:
    Xd, Yd = diag[g]
    Ps = (predict_mlp_static(mlp_static, Xd) > 0.5)
    Pm = (predict_mlp(mlp, Xd) > 0.5)
    Pc = (predict_conv(conv, Xd) > 0.5)
    mask_b = Yd == 1
    static_b.append((Ps[mask_b] == 1).mean())
    mlp_b.append((Pm[mask_b] == 1).mean())
    conv_b.append((Pc[mask_b] == 1).mean())

fig, ax = plt.subplots(figsize=(9*.7, 3*.7), constrained_layout=True)
x = np.arange(len(groups))
w = 0.27
ax.bar(x - w, static_b, width=w, label="MLP (static)", color="#9467bd")
ax.bar(x,     mlp_b,    width=w, label="MLP (window)", color="#d62728")
ax.bar(x + w, conv_b,   width=w, label="Conv1D (window)", color="#1f77b4")
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylim(0, 1.05); ax.set_ylabel("recall on class-B timesteps")
ax.axhline(1.0, color="black", linewidth=0.5, linestyle=":")
ax.legend(loc='lower right')
fig.set_dpi(200)
ax.set_title("How often does the model correctly mark a class-B timestep?")
plt.show()

Two inductive-bias steps:

**Static MLP -> windowed MLP (information bottleneck).** The static MLP sees one timestep at a time, so it is structurally blind to pattern B (an oscillation only visible across time). It does fine on `only_A` (the mean shift is local) and fails on `only_B`. Giving the MLP a window unlocks pattern B *in principle*.

**Windowed MLP -> Conv1D (translational symmetry).** Both window models receive the same input, so the windowed MLP is expressive enough to detect both patterns. In practice the Conv1D's translational symmetry is a much stronger inductive bias.